## Домашнее задание 1. От вопроса к набору данных

### Исследовательский вопрос: как менялась тональность текстов Пушкина со временем?

Известно, что у некоторых авторов (например, у Льва Толстого) в течение жизни произведения довольно сильно меняют свое настроение - становятся более негативными или позитивными. Цель этой работы - увидеть, есть ли такое у Пушкина.

Для создания корпуса было использовано MediaWiki API, так как там собрано очень большое количество классических текстов. Также там уже существуют две коллекции стихов Пушкина:

* Ранний период (1809–1825): https://ru.wikisource.org/wiki/Стихотворения_Пушкина_1809%E2%80%941825

* Поздний период (1826–1836): https://ru.wikisource.org/wiki/Стихотворения_Пушкина_1826%E2%80%941836


Эти два периода я и решила сравнивать. Для определения настроения использовалась модель seara/rubert-tiny2-russian-sentiment.


### Загрузка текстов

In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# страницы с текстами
period_pages = {
    "Ранний период (1809–1825)": "https://ru.wikisource.org/wiki/Стихотворения_Пушкина_1809%E2%80%941825",
    "Поздний период (1826–1836)": "https://ru.wikisource.org/wiki/Стихотворения_Пушкина_1826%E2%80%941836"}

headers = {"User-Agent": "DH-scraping (https://github.com/elenatutubalina/mini_DH)"}
poems = []

# проходим по страницам
for period, url in period_pages.items():
    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # содержимое страницы
    content = soup.select_one("#mw-content-text .mw-parser-output")

    if content is None:
        print("Не удалось найти содержимое:", period)
        continue

    # список
    for li in content.find_all("li"):
        # название раздела
        heading = li.find_previous("h4")
        if heading is None:
            continue

        section = heading.get_text(" ", strip=True)

        # есть также "Dubia" и "Отрывки"
        if "Стихотворения" in section or "Поэмы" in section:
          link = li.find("a", href=True)

          if link is None:
              continue

          href = link["href"]

          if not href.startswith("/wiki/"):
              continue

          poems.append({
              "период": period,
              "раздел": section,
              "название": link.get_text(" ", strip=True),
              "ссылка": urljoin(url, href)})

# датафрейм
poems_df = pd.DataFrame(poems)

poems_df = poems_df.drop_duplicates(subset="ссылка").reset_index(drop=True)

print("Всего найдено произведений:", len(poems_df))
print("\nКоличество по периодам:")
print(poems_df["период"].value_counts())

poems_df


Всего найдено произведений: 776

Количество по периодам:
период
Ранний период (1809–1825)     438
Поздний период (1826–1836)    338
Name: count, dtype: int64


,период,раздел,название,ссылка
0,Ранний период (1809–1825),Стихотворения 1809—1812 г.,"«Dis moi, pourquoi l’Escamoteur…»","https://ru.wikisource.org/wiki/Dis_moi,_pourqu..."
1,Ранний период (1809–1825),Стихотворения 1809—1812 г.,"«Je chante ce combat, que Toly remporta…»",https://ru.wikisource.org/wiki/Je_chante_ce_co...
2,Ранний период (1809–1825),Стихотворения 1813 г.,,https://ru.wikisource.org/wiki/%D0%9A_%D0%9D%D...
3,Ранний период (1809–1825),Стихотворения 1813 г.,"Монах («Хочу воспеть, как дух нечистый Ада…»)",https://ru.wikisource.org/wiki/%D0%9C%D0%BE%D0...
4,Ранний период (1809–1825),Стихотворения 1813 г.,Несчастие Клита («Внук Тредьяковского Клит гек...,https://ru.wikisource.org/wiki/%D0%9D%D0%B5%D1...
...,...,...,...,...
771,Поздний период (1826–1836),Поэмы,Полтава,https://ru.wikisource.org/wiki/%D0%9F%D0%BE%D0...
772,Поздний период (1826–1836),Поэмы,Тазит,https://ru.wikisource.org/wiki/%D0%A2%D0%B0%D0...
773,Поздний период (1826–1836),Поэмы,Домик в Коломне,https://ru.wikisource.org/wiki/%D0%94%D0%BE%D0...
774,Поздний период (1826–1836),Поэмы,Анджело,https://ru.wikisource.org/wiki/%D0%90%D0%BD%D0...


смотрим какие разделы попали

In [3]:
for period in poems_df["период"].unique():
    print(period)
    section_counts = (poems_df[poems_df["период"] == period]["раздел"].value_counts())
    print(section_counts)

Ранний период (1809–1825)
раздел
Стихотворения 1825 г.                  51
Стихотворения 1821 г.                  49
Стихотворения 1824 г.                  41
Стихотворения 1816 г.                  35
Стихотворения 1819 г.                  32
Стихотворения 1815 г.                  28
Стихотворения 1814 г.                  26
Стихотворения 1822 г.                  26
Стихотворения 1823 г.                  25
Стихотворения 1813—1817 гг.            22
Стихотворения 1817 г. (Лицей)          19
Стихотворения 1818 г.                  18
Стихотворения 1820 г.                  18
Стихотворения 1817—1820 гг.            16
Стихотворения 1817 г. (После Лицея)    12
Поэмы                                   8
Стихотворения 1817—1825 гг.             7
Стихотворения 1813 г.                   3
Стихотворения 1809—1812 г.              2
Name: count, dtype: int64
Поздний период (1826–1836)
раздел
Стихотворения 1828 г.         51
Стихотворения 1829 г.         50
Стихотворения 1830 г.         43
Стихотворе

Видно, что в поздний период также попадают "Стихотворения разных годов, 1814—1836 г.", что плохо, поскольку они написаны не только в этот период. Остальные все разделы нормальные, их можно оставить, удаляем только "Стихотворения разных годов, 1814—1836 г."


In [4]:
poems_df = poems_df[~((poems_df["период"] == "Поздний период (1826–1836)") & (poems_df["раздел"] == "Стихотворения разных годов, 1814—1836\xa0г."))].reset_index(drop=True)

print("Осталось произведений:", len(poems_df))
print("\nКоличество по периодам:")
print(poems_df["период"].value_counts())

Осталось произведений: 768

Количество по периодам:
период
Ранний период (1809–1825)     438
Поздний период (1826–1836)    330
Name: count, dtype: int64


теперь можно уже собирать тексты стихов

In [5]:
import time
import re

texts = []
errors = []

# проходим по всем текстам
for i, row in poems_df.iterrows():

    try:
        response = requests.get(row["ссылка"], headers=headers)
        soup = BeautifulSoup(response.text, "html.parser")

        # находим текст
        content = soup.select_one("#mw-content-text .mw-parser-output")

        if content is None:
            texts.append("")
            errors.append(row["ссылка"])
            continue

        # удаляем служебные элементы
        for element in content.select(".ws-noexport, .mw-editsection, .catlinks, .navbox, script, style"):
            element.decompose()

        # текст
        text = content.get_text("\n", strip=True)

        # нормализуем текст
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n+", "\n", text)

        texts.append(text)

        if not text:
            errors.append(row["ссылка"])

    except:
        texts.append("")
        errors.append(row["ссылка"])

    time.sleep(0.3)

# добавляем в дф
poems_df["текст"] = texts

print("Всего стихотворений:", len(poems_df))
print("Не удалось загрузить:", len(errors))
print("Успешно загружено:", len(poems_df) - len(errors))

Всего стихотворений: 768
Не удалось загрузить: 105
Успешно загружено: 663


тексты, которые не скачались:

In [6]:
# таблица с ошибками
errors_df = pd.DataFrame(errors, columns=["ссылка"])

# находим произведения с пустым текстом
empty_poems = poems_df[poems_df["текст"].isna() | (poems_df["текст"].str.strip() == "")].copy()

# добавляем информацию об ошибках
problem_poems = empty_poems.merge(errors_df, on="ссылка", how="left")

print("Всего произведений:", len(poems_df))
print("Требуют восстановления:", len(problem_poems))

problem_poems[["период", "раздел", "название", "ссылка"]]

Всего произведений: 768
Требуют восстановления: 105


,период,раздел,название,ссылка
0,Ранний период (1809–1825),Стихотворения 1815 г.,,https://ru.wikisource.org/wiki/%D0%93%D0%BE%D1...
1,Ранний период (1809–1825),Стихотворения 1815 г.,,https://ru.wikisource.org/wiki/%D0%92%D0%BE%D0...
2,Ранний период (1809–1825),Стихотворения 1815 г.,,https://ru.wikisource.org/wiki/%D0%98%D0%B7%D0...
3,Ранний период (1809–1825),Стихотворения 1815 г.,,https://ru.wikisource.org/wiki/%D0%9A_%D0%9B%D...
4,Ранний период (1809–1825),Стихотворения 1815 г.,,https://ru.wikisource.org/wiki/%D0%91%D0%B0%D1...
...,...,...,...,...
100,Ранний период (1809–1825),Стихотворения 1817—1820 гг.,,https://ru.wikisource.org/wiki/%D0%A2%D1%8B_%D...
101,Ранний период (1809–1825),Стихотворения 1817—1820 гг.,,https://ru.wikisource.org/wiki/%D0%97%D0%B0%D0...
102,Ранний период (1809–1825),Стихотворения 1817—1820 гг.,,https://ru.wikisource.org/wiki/%D0%94%D0%BE%D0...
103,Ранний период (1809–1825),Стихотворения 1817—1820 гг.,,https://ru.wikisource.org/wiki/%D0%9A_%D0%BF%D...


Перейдя по этим ссылкам, судя по всему, к ним добавляется /ДО в конце из-за значка дореволюционной орфографии в списке. Проверяем есть ли этот суффикс у всех ссылок

In [7]:
from urllib.parse import unquote

# декодируем ссылки чтобы кириллица была видна
errors_df["декодированная_ссылка"] = errors_df["ссылка"].apply(lambda x: unquote(str(x)))

# ищем /ДО в ссылке
do_errors = errors_df[errors_df["декодированная_ссылка"].str.contains("/ДО", case=False, na=False)]

print("Количество ошибок, связанных с /ДО:", len(do_errors))

do_errors[["ссылка", "декодированная_ссылка"]]

Количество ошибок, связанных с /ДО: 105


,ссылка,декодированная_ссылка
0,https://ru.wikisource.org/wiki/%D0%93%D0%BE%D1...,https://ru.wikisource.org/wiki/Городок_(Пушкин...
1,https://ru.wikisource.org/wiki/%D0%92%D0%BE%D0...,https://ru.wikisource.org/wiki/Вода_и_вино_(Пу...
2,https://ru.wikisource.org/wiki/%D0%98%D0%B7%D0...,https://ru.wikisource.org/wiki/Измены_(Пушкин)/ДО
3,https://ru.wikisource.org/wiki/%D0%9A_%D0%9B%D...,https://ru.wikisource.org/wiki/К_Лицинию_(С_ла...
4,https://ru.wikisource.org/wiki/%D0%91%D0%B0%D1...,https://ru.wikisource.org/wiki/Батюшкову_(Пушк...
...,...,...
100,https://ru.wikisource.org/wiki/%D0%A2%D1%8B_%D...,https://ru.wikisource.org/wiki/Ты_и_я_(Пушкин)/ДО
101,https://ru.wikisource.org/wiki/%D0%97%D0%B0%D0...,https://ru.wikisource.org/wiki/Записка_к_Жуков...
102,https://ru.wikisource.org/wiki/%D0%94%D0%BE%D0...,https://ru.wikisource.org/wiki/Добрый_человек_...
103,https://ru.wikisource.org/wiki/%D0%9A_%D0%BF%D...,https://ru.wikisource.org/wiki/К_портрету_Дель...


Попробуем теперь убрать это /ДО в конце ссылки и скачаем заново

In [8]:
recovered = 0
recovery_errors = []

# находим произведения с пустым текстом
indices = poems_df[ poems_df["текст"].isna() | (poems_df["текст"].str.strip() == "")].index

print("Произведений для восстановления:", len(indices))

# повторно загружаем пустые тексты
for idx in indices:

    row = poems_df.loc[idx]
    url = unquote(row["ссылка"])

    # убираем /ДО в конце ссылки
    if url.endswith("/ДО"):
        url = url[:-3]

    try:
        response = requests.get(url, headers=headers, timeout=20)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        # находим текст
        content = soup.select_one("#mw-content-text .mw-parser-output")

        if content is None:
            raise ValueError("Текст не найден")

        # удаляем служебные элементы
        for element in content.select(".ws-noexport, .mw-editsection, .catlinks, .navbox, script, style"):
            element.decompose()

        # текст
        text = content.get_text("\n", strip=True)

        # нормализуем текст
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n+", "\n", text)

        if not text.strip():
            raise ValueError("Пустой текст")

        # сохраняем текст в таблицу
        poems_df.at[idx, "текст"] = text
        recovered += 1

    except Exception as e:
        recovery_errors.append({
            "индекс": idx,
            "название": row["название"],
            "ссылка": url,
            "ошибка": str(e)})

    time.sleep(0.3)

# выводим результаты
print("\nВосстановление завершено")
print("Успешно загружено:", recovered)
print("Не удалось загрузить:", len(recovery_errors))
print("Всего текстов с содержимым:", poems_df["текст"].fillna("").str.strip().ne("").sum())

Произведений для восстановления: 105

Восстановление завершено
Успешно загружено: 105
Не удалось загрузить: 0
Всего текстов с содержимым: 768


все получилось восстановить. теперь посмотрим на статистику периодов по длине текстов

In [9]:
# статистика длины текстов
poems_df["длина"] = poems_df["текст"].str.len()

print("Статистика длины текстов:")
display(poems_df.groupby("период")["длина"].describe())

print("\nСамые короткие загруженные тексты:")
display(poems_df[poems_df["длина"] > 0].sort_values("длина")[["период", "название", "длина", "текст"]].head())

print("\nСамые длинные тексты:")
display(poems_df.sort_values("длина", ascending=False)[["период", "название", "длина", "текст"]].head())

Статистика длины текстов:


,count,mean,std,min,25%,50%,75%,max
период,,,,,,,,
Поздний период (1826–1836),330.0,1671.760606,2380.280136,69.0,508.5,900.5,1886.75,23174.0
Ранний период (1809–1825),438.0,1950.340183,4241.651697,54.0,575.0,1071.0,1935.25,75305.0



Самые короткие загруженные тексты:


,период,название,длина,текст
432,Ранний период (1809–1825),Кавказский пленник,54,Посвящение\nЧасть первая\nЧасть вторая\nЭпилог...
543,Поздний период (1826–1836),"«Покойник, автор сухощавый…»",69,"* * *\nПокойник, автор сухощавый,\nПисал для д..."
542,Поздний период (1826–1836),«Лищинский околел — отечеству беда!..»,88,* * *\nЛищинский околел — отечеству беда!\nКня...
756,Поздний период (1826–1836),«Забыв и рощу и свободу…»,107,"* * *\nЗабыв и рощу и свободу,\nНевольный чижи..."
129,Ранний период (1809–1825),Эпиграмма на смерть стихотворца («Покойник Кли...,115,* * *\nПокойник Клит в раю не будет:\nТворил о...



Самые длинные тексты:


,период,название,длина,текст
431,Ранний период (1809–1825),Руслан и Людмила,75305,"Руслан и Людмила\nПоэма\nПосвящение\nДля вас, ..."
766,Поздний период (1826–1836),Анджело,23174,Анджело\nЧасть первая\nI\nВ одном из городов И...
765,Поздний период (1826–1836),Домик в Коломне,21458,I. «Четырёстопный ямб мне надоел…»\nII. «А что...
433,Ранний период (1809–1825),Гавриилиада,21228,ГАВРИИЛИАДА\nПОЭМА\n1821\n‎\nВоистину еврейки ...
436,Ранний период (1809–1825),Цыганы,17131,Цыганы\nЦыганы шумною толпой\nПо Бессарабии ко...


### Анализ тона

In [ ]:
pip install razdel transformers torch

Анализировать лучше каждое предложение и для каждого стихотворения считать количество позитивный vs нейтральный vs негативный (в предыдущей ячейке видно, что есть большое количество длинных текстов, где тональность наверняка много раз меняется между предложениями)

In [10]:
from razdel import sentenize

sentences = []
for idx, row in poems_df.iterrows():
    text = row["текст"]
    if not isinstance(text, str) or not text.strip():
        continue

    # бьем стихотворение на предложения
    for number, sentence in enumerate(sentenize(text), start=1):
        sentence_text = sentence.text.strip()
        if not sentence_text:
            continue

        sentences.append({
            "id_стихотворения": idx,
            "период": row["период"],
            "название": row["название"],
            "номер_предложения": number,
            "предложение": sentence_text})

sentences_df = pd.DataFrame(sentences)

print("Всего предложений:", len(sentences_df))
print("Всего стихотворений:", sentences_df["id_стихотворения"].nunique())

sentences_df.head()

Всего предложений: 19157
Всего стихотворений: 768


,id_стихотворения,период,название,номер_предложения,предложение
0,0,Ранний период (1809–1825),"«Dis moi, pourquoi l’Escamoteur…»",1,"* * *\nDis moi, pourquoi l’Escamoteur\nEst-il ..."
1,0,Ранний период (1809–1825),"«Dis moi, pourquoi l’Escamoteur…»",2,Hélas ! c’est que le pauvre auteur\nL’escamota...
2,0,Ранний период (1809–1825),"«Dis moi, pourquoi l’Escamoteur…»",3,1809-1811?
3,0,Ранний период (1809–1825),"«Dis moi, pourquoi l’Escamoteur…»",4,"↑\nСкажите мне, почему «Похититель»\nОсвистан ..."
4,0,Ранний период (1809–1825),"«Dis moi, pourquoi l’Escamoteur…»",5,"Увы, потому что бедный автор\nПохитил его у Мо..."


проверим, что модель вообще нормально работает:

In [11]:
from transformers import pipeline

model = pipeline(model="seara/rubert-tiny2-russian-sentiment")

print(model("Я обожаю программирование!"))
print(model("Я ненавижу программирование!"))
print(model("Я иду на программирование!"))

Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

[{'label': 'positive', 'score': 0.9903780221939087}]
[{'label': 'negative', 'score': 0.8729127645492554}]
[{'label': 'neutral', 'score': 0.8957920670509338}]


применяем к предложениям стихов

In [12]:
texts = sentences_df["предложение"].tolist()

# анализ пачками
predictions = model(texts, batch_size=32)

sentences_df["результат"] = [prediction["label"] for prediction in predictions]
sentences_df["уверенность"] = [prediction["score"] for prediction in predictions]

sentences_df.head(10)

,id_стихотворения,период,название,номер_предложения,предложение,результат,уверенность
0,0,Ранний период (1809–1825),"«Dis moi, pourquoi l’Escamoteur…»",1,"* * *\nDis moi, pourquoi l’Escamoteur\nEst-il ...",negative,0.656017
1,0,Ранний период (1809–1825),"«Dis moi, pourquoi l’Escamoteur…»",2,Hélas ! c’est que le pauvre auteur\nL’escamota...,negative,0.670122
2,0,Ранний период (1809–1825),"«Dis moi, pourquoi l’Escamoteur…»",3,1809-1811?,neutral,0.858661
3,0,Ранний период (1809–1825),"«Dis moi, pourquoi l’Escamoteur…»",4,"↑\nСкажите мне, почему «Похититель»\nОсвистан ...",negative,0.762607
4,0,Ранний период (1809–1825),"«Dis moi, pourquoi l’Escamoteur…»",5,"Увы, потому что бедный автор\nПохитил его у Мо...",negative,0.790458
5,0,Ранний период (1809–1825),"«Dis moi, pourquoi l’Escamoteur…»",6,(\nФранц.\n),neutral,0.924148
6,0,Ранний период (1809–1825),"«Dis moi, pourquoi l’Escamoteur…»",7,Примечания\nПечатается по рукописи «Воспоминан...,neutral,0.943343
7,0,Ранний период (1809–1825),"«Dis moi, pourquoi l’Escamoteur…»",8,Впервые опубликовано П. В. Анненковым (по той ...,neutral,0.863740
8,0,Ранний период (1809–1825),"«Dis moi, pourquoi l’Escamoteur…»",9,"Стихи записаны в 1851 г., т. е. спустя сорок с...",neutral,0.693304
9,0,Ранний период (1809–1825),"«Dis moi, pourquoi l’Escamoteur…»",10,"Эти французские стихи, по-видимому, первое, чт...",neutral,0.749124


смотрим статистику

In [13]:
# количество предложений каждой тональности
sentiment_counts = pd.crosstab(sentences_df["период"], sentences_df["результат"])
sentiment_counts

результат,negative,neutral,positive
период,,,
Поздний период (1826–1836),1012,5242,708
Ранний период (1809–1825),1623,9033,1539


In [16]:
sentiment_percentages = (sentiment_counts.div(sentiment_counts.sum(axis=1), axis=0) * 100).round(2)
sentiment_percentages

результат,negative,neutral,positive
период,,,
Поздний период (1826–1836),14.54,75.29,10.17
Ранний период (1809–1825),13.31,74.07,12.62


Ожидаемо, в обоих периодах больше всего нейтральных предложений. Однако в раннем периоде достаточно близкое количество позитивных и негативных предложений, а в позднем негативных ощутимо больше. Из этого можно сделать вывод, что в поздний период творчества Пушкина его стихи стали немного более негативными

In [17]:
# сохраняем таблицу со стихотворениями
poems_df.to_csv("data.csv", index=False, encoding="utf-8")
